In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations

# =========================================================
# 0) 실행 환경 / 입출력 경로
# =========================================================
# [지하철용 수정] 기존 버스/이동 데이터 입력 경로 대신 지하철 30분단위 이용 통계 CSV를 사용합니다.
DATA_FILE = Path("8SSIBLE_SG_DATA/서울시 지하철 30분단위 이용 통계.csv")
OUTPUT_DIR = Path("output_subway")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 노트북 실행 위치가 repo root 또는 branch_SG 둘 중 어디여도 동작하도록 처리
def resolve_path(path: Path) -> Path:
    candidates = [Path.cwd() / path, Path.cwd() / "branch_SG" / path]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")

INPUT_PATH = resolve_path(DATA_FILE)

# 필요하면 분석 연도를 지정하세요. None이면 전체 연도를 사용합니다.
YEAR_FROM = None
YEAR_TO = None

# [지하철용 수정] 기존 자치구/행정동 이동 집계 대신 역 또는 호선 단위로 분석합니다.
# 역별 분석이면 ['호선명', '역ID', '역명'], 호선별 분석이면 ['호선명']으로 변경
GROUP_COLS = ["호선명", "역ID", "역명"]

# [지하철용 수정] 이동시간/유동인구 기반 피처를 지하철 이용량 기반 피처로 교체했습니다.
# 문헌/도메인 기반 초기 가중치
LITERATURE_WEIGHTS = {
    "총이용인원": 0.45,
    "평균30분이용인원": 0.25,
    "혼잡피크이용인원": 0.20,
    "출퇴근이용비중": 0.10,
}

FEATURE_COLS = list(LITERATURE_WEIGHTS.keys())
SCORE_COL = "지하철이용스트레스점수"


# =========================================================
# 1) 공통 함수
# =========================================================
# [지하철용 수정] 지하철 CSV 전용 로더입니다. 한글 CSV 인코딩을 순차적으로 시도합니다.
def load_subway_data(path: Path) -> pd.DataFrame:
    for encoding in ("utf-8-sig", "utf-8", "cp949", "euc-kr"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"파일을 읽을 수 없습니다: {path}")

def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    if s.max() == s.min():
        return pd.Series(0.0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

def normalize_weights(weights: dict) -> dict:
    total = sum(weights.values())
    if total == 0:
        return {k: 1 / len(weights) for k in weights}
    return {k: v / total for k, v in weights.items()}

def rank_spearman(df_a, df_b, key_cols, col=SCORE_COL):
    a = df_a[key_cols + [col]].rename(columns={col: "a"})
    b = df_b[key_cols + [col]].rename(columns={col: "b"})
    m = a.merge(b, on=key_cols, how="inner")
    if len(m) < 2:
        return np.nan
    return m["a"].rank().corr(m["b"].rank(), method="pearson")

def topk_overlap(df_a, df_b, key_cols, k=10, col=SCORE_COL):
    a = df_a.sort_values(col, ascending=False).head(k)[key_cols].astype(str).agg("|".join, axis=1)
    b = df_b.sort_values(col, ascending=False).head(k)[key_cols].astype(str).agg("|".join, axis=1)
    return len(set(a) & set(b)) / k if k > 0 else np.nan


# =========================================================
# 2) 지하철 데이터 전처리
# =========================================================
# [지하철용 수정] 기존 start_dt/arv_dt/start_emd/arv_emd/population 컬럼 전처리를
# 지하철 원천 컬럼(년, 월, 일, 시간, 분, 역ID, 호선명, 역명, 승하차인원) 기준으로 바꿨습니다.
def preprocess_subway(raw_df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {
        "년(YEAR)": "연도",
        "월(MONTH)": "월",
        "일(DAY)": "일",
        "시간(HOUR)": "시간",
        "분_30분단위(HALF_HOUR)": "분",
        "역ID(STATION_ID)": "역ID",
        "호선명(LINE_NM)": "호선명",
        "역명(STATION_NM)": "역명",
        "승차인원(GETON_CNT)": "승차인원",
        "하차인원(GETOFF_CNT)": "하차인원",
    }

    missing = [c for c in rename_map if c not in raw_df.columns]
    if missing:
        raise KeyError(f"필수 컬럼이 없습니다: {missing}")

    df = raw_df[list(rename_map)].rename(columns=rename_map).copy()

    for col in ["연도", "월", "일", "시간", "분", "승차인원", "하차인원"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["역ID"] = df["역ID"].astype(str).str.strip().str.zfill(4)
    df["호선명"] = df["호선명"].astype(str).str.strip()
    df["역명"] = df["역명"].astype(str).str.strip()
    df = df.dropna(subset=["연도", "월", "일", "시간", "분", "승차인원", "하차인원"]).copy()

    df["일시"] = pd.to_datetime(
        dict(year=df["연도"], month=df["월"], day=df["일"], hour=df["시간"], minute=df["분"]),
        errors="coerce",
    )
    df = df.dropna(subset=["일시"]).copy()
    df["연도"] = df["일시"].dt.year
    df["월"] = df["일시"].dt.month
    df["요일"] = df["일시"].dt.day_name()
    # [지하철용 수정] 승차+하차를 총 이용량으로 보고, 30분 단위 혼잡도를 계산할 기본값으로 사용합니다.
    df["총이용인원"] = df["승차인원"] + df["하차인원"]
    # [지하철용 수정] 기존 H 포함 이동유형 필터 대신 출퇴근 시간대 비중을 별도 피처로 만듭니다.
    df["출퇴근시간"] = df["시간"].between(7, 9) | df["시간"].between(18, 20)

    if YEAR_FROM is not None:
        df = df[df["연도"] >= YEAR_FROM]
    if YEAR_TO is not None:
        df = df[df["연도"] <= YEAR_TO]

    return df.reset_index(drop=True)


# =========================================================
# 3) 역/호선별 피처 생성
# =========================================================
# [지하철용 수정] 기존 도착 자치구별 이동량/이동시간 집계 대신 역별 승하차량과 피크 이용량을 집계합니다.
def make_features(df: pd.DataFrame, group_cols=GROUP_COLS) -> pd.DataFrame:
    base = (
        df.groupby(group_cols, dropna=False)
        .agg(
            총승차인원=("승차인원", "sum"),
            총하차인원=("하차인원", "sum"),
            총이용인원=("총이용인원", "sum"),
            평균30분이용인원=("총이용인원", "mean"),
            혼잡피크이용인원=("총이용인원", "max"),
            관측건수=("총이용인원", "size"),
        )
        .reset_index()
    )

    commute = (
        df[df["출퇴근시간"]]
        .groupby(group_cols, dropna=False)["총이용인원"]
        .sum()
        .reset_index(name="출퇴근이용인원")
    )

    f = base.merge(commute, on=group_cols, how="left")
    f["출퇴근이용인원"] = f["출퇴근이용인원"].fillna(0)
    f["출퇴근이용비중"] = np.where(f["총이용인원"] > 0, f["출퇴근이용인원"] / f["총이용인원"], 0)
    return f

def make_features_by_year(df: pd.DataFrame, group_cols=GROUP_COLS) -> pd.DataFrame:
    parts = []
    for year, part in df.groupby("연도"):
        f = make_features(part, group_cols)
        f.insert(0, "연도", year)
        parts.append(f)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


# =========================================================
# 4) 가중치 및 점수 산출
# =========================================================
# [지하철용 수정] 점수 산출 구조(PCA/CRITIC/문헌기반)는 유지하되, 입력 피처만 지하철 피처로 교체했습니다.
def pca_weights(feature_df: pd.DataFrame) -> dict:
    X = feature_df[FEATURE_COLS].astype(float)
    X = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
    X = X.fillna(0.0)
    cov = np.cov(X.values, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    load = np.abs(eigvecs[:, np.argmax(eigvals)])
    return normalize_weights(dict(zip(FEATURE_COLS, load)))

def critic_weights(feature_df: pd.DataFrame) -> dict:
    Z = feature_df[FEATURE_COLS].astype(float)
    Z = (Z - Z.min()) / (Z.max() - Z.min())
    Z = Z.fillna(0.0)
    std = Z.std(ddof=0)
    corr = Z.corr().fillna(0.0)
    cvals = {col: std[col] * (1 - corr[col]).sum() for col in FEATURE_COLS}
    return normalize_weights(cvals)

def score_with_weights(feature_df: pd.DataFrame, weights: dict, model_name: str) -> pd.DataFrame:
    d = feature_df.copy()
    w = normalize_weights(weights)
    score = 0
    for col in FEATURE_COLS:
        d[f"{col}_norm"] = minmax(d[col])
        score += d[f"{col}_norm"] * w[col]

    d[SCORE_COL] = score * 100
    d["모형"] = model_name
    for col in FEATURE_COLS:
        d[f"가중치_{col}"] = w[col]

    return d.sort_values(SCORE_COL, ascending=False).reset_index(drop=True)


# =========================================================
# 5) 전체 실행
# =========================================================
# [지하철용 수정] 전체 파이프라인 이름과 출력 파일명을 지하철 분석 결과에 맞게 정리했습니다.
def run_pipeline_all(raw_df: pd.DataFrame):
    subway = preprocess_subway(raw_df)
    features = make_features(subway, GROUP_COLS)

    w_lit = normalize_weights(LITERATURE_WEIGHTS)
    w_pca = pca_weights(features)
    w_critic = critic_weights(features)

    scores = pd.concat(
        [
            score_with_weights(features, w_lit, "문헌기반"),
            score_with_weights(features, w_pca, "PCA기반"),
            score_with_weights(features, w_critic, "CRITIC기반"),
        ],
        ignore_index=True,
    )

    base = scores[scores["모형"] == "문헌기반"]
    sensitivity_rows = []
    for model in scores["모형"].unique():
        cur = scores[scores["모형"] == model]
        sensitivity_rows.append(
            {
                "비교모형": model,
                "스피어만순위상관_문헌대비": rank_spearman(base, cur, GROUP_COLS),
                "Top10중복률_문헌대비": topk_overlap(base, cur, GROUP_COLS, k=10),
            }
        )
    sensitivity = pd.DataFrame(sensitivity_rows)

    yearly_features = make_features_by_year(subway, GROUP_COLS)
    yearly_scores_parts = []
    if not yearly_features.empty:
        for year, part in yearly_features.groupby("연도"):
            sy = score_with_weights(part.drop(columns=["연도"]), w_lit, f"문헌기반_{year}")
            sy.insert(0, "연도", year)
            yearly_scores_parts.append(sy[["연도"] + GROUP_COLS + [SCORE_COL]])

    yearly_scores = pd.concat(yearly_scores_parts, ignore_index=True) if yearly_scores_parts else pd.DataFrame()

    stability_rows = []
    if not yearly_scores.empty:
        years = sorted(yearly_scores["연도"].unique())
        for y1, y2 in combinations(years, 2):
            a = yearly_scores[yearly_scores["연도"] == y1]
            b = yearly_scores[yearly_scores["연도"] == y2]
            stability_rows.append({"연도쌍": f"{y1}-{y2}", "스피어만순위상관": rank_spearman(a, b, GROUP_COLS)})
    stability = pd.DataFrame(stability_rows)

    meta = pd.DataFrame(
        [
            {
                "입력파일": str(INPUT_PATH),
                "분석단위": ", ".join(GROUP_COLS),
                "원본행수": len(raw_df),
                "전처리후행수": len(subway),
                "분석연도": f"{subway['연도'].min()}~{subway['연도'].max()}" if len(subway) else "",
            }
        ]
    )

    return {
        "전처리_지하철데이터": subway,
        "피처": features,
        "모형별_점수결과": scores,
        "민감도분석": sensitivity,
        "연도별_점수": yearly_scores,
        "연도안정성분석": stability,
        "메타정보": meta,
    }


print("[START] 지하철 데이터 로드")
raw_df = load_subway_data(INPUT_PATH)
print(f"[INFO] 원본 행 수: {len(raw_df):,}")

outputs = run_pipeline_all(raw_df)

outputs["메타정보"].to_csv(OUTPUT_DIR / "00_메타정보.csv", index=False, encoding="utf-8-sig")
outputs["전처리_지하철데이터"].to_csv(OUTPUT_DIR / "01_전처리_지하철데이터.csv", index=False, encoding="utf-8-sig")
outputs["피처"].to_csv(OUTPUT_DIR / "02_피처.csv", index=False, encoding="utf-8-sig")
outputs["모형별_점수결과"].to_csv(OUTPUT_DIR / "03_모형별_점수결과.csv", index=False, encoding="utf-8-sig")
outputs["민감도분석"].to_csv(OUTPUT_DIR / "04_민감도분석.csv", index=False, encoding="utf-8-sig")
outputs["연도별_점수"].to_csv(OUTPUT_DIR / "05_연도별_점수.csv", index=False, encoding="utf-8-sig")
outputs["연도안정성분석"].to_csv(OUTPUT_DIR / "06_연도안정성분석.csv", index=False, encoding="utf-8-sig")

print("[DONE] 지하철 분석 파일 저장 완료")
print(f"[INFO] 저장 경로: {OUTPUT_DIR.resolve()}")
outputs["모형별_점수결과"].head(10)
